<a href="https://colab.research.google.com/github/jorge-alvarenga/olx/blob/main/OLX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Coletando Dados

Vamos criar um Web Scraping para coletar dados do site olx para montar uma base de dados para que possamos realizar uma anilse de dados usando sql e depois criar um Dashboard com esse dados coletados.

In [3]:
!pip install unidecode

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 235 kB 8.6 MB/s 


In [4]:
# Importando bibliotecas
from unidecode import unidecode
from urllib.request import urlopen, urlretrieve, Request
from bs4 import BeautifulSoup
import pandas as pd
import math
import requests

# Declarando variável cards
cards = []

# Obtendo o HTML
url = 'https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios'

headers = {'User-Agent': 'Mozilla/5.0'}

try:
    req = Request(url, headers = headers)
    response = urlopen(req)
    
except HTTPError as e:
    print(e.status, e.reason)
    
except URLError as e:
    print(e.reason)

html = response.read().decode('utf-8')
soup = BeautifulSoup(html, 'html.parser')

In [5]:
marcas =[]
for marca in soup.find('select', {"class": "g98wuw-1 jSSsDV"}).findAll('option'):
    marca = marca.text
    marca = marca.replace(' ','-')
    marca = marca.replace('---','-')
    marca = marca.lower()
    marcas.append(marca)
  


In [6]:
### captura de modelos atraves das marcas###
dataset = []

for marca in marcas:
    url = 'https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/'+marca
    try:
        req = Request(url, headers = headers)
        response = urlopen(req)
    
    except HTTPError as e:
        print(e.status, e.reason)
        continue
    except URLError as e:
        print(e.reason)
        continue

    html = response.read().decode('utf-8')
    soup = BeautifulSoup(html, 'html.parser')
    print(url)
    try:
        for modelo in soup.findAll('div', {"class": "g98wuw-0 NTTCv"})[1].findAll('option'):
            modelos = {}
            modelos['modelo'] = modelo.text
            modelos['marca'] = marca
            modelo = modelo.text
            modelo = modelo.replace(' ','-')
            modelo = modelo.replace('---','-')
            modelo = modelo.lower()
            modelo = unidecode(modelo)

            url_modelo = url+'/'+modelo
            print(url_modelo)
            try:
                req_modelo = Request(url_modelo, headers = headers)
                response_modelo = urlopen(req_modelo)

            except HTTPError as e:
                print(e.status, e.reason)
                continue 
            except URLError as e:
                print(e.reason)
                continue
            html_modelo = response_modelo.read().decode('utf-8')
            soup_modelo = BeautifulSoup(html_modelo, 'html.parser')
            for estado in soup_modelo.find('div', {"sc-hmzhuo sc-1a202fr-0 loGutE sc-jTzLTM iwtnNi"}).findAll('div', {"class": "sc-hmzhuo kylvO sc-jTzLTM iwtnNi"}):
                estado = estado.text.split(',')
                modelos[estado[0]] = estado[1]
            dataset.append(modelos)
    except:
        continue

https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/marca
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/marca/modelo
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/modelo
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/aircross
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/ax
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/berlingo
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/bx
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/c3
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/c4
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/c4-cactus
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/c5
https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/citroen/c6
https://www.olx.com.b

In [11]:
from google.colab import drive
drive.mount ('/drive')
dataset_cursos = pd.DataFrame(dataset)
dataset_cursos.to_csv('/drive/MyDrive/Projetos/OLX/meu_arquivo_csv.csv', index = False, encoding = 'utf-8-sig')
dataset_cursos

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


,modelo,marca,Acre,Alagoas,Amapá,Amazonas,Bahia,Ceará,Distrito Federal,Espírito Santo,...,Piauí,Rio de Janeiro,Rio Grande do Norte,Rio Grande do Sul,Rondônia,Roraima,Santa Catarina,São Paulo,Sergipe,Tocantins
0,Modelo,marca,2.229,7.561,1.067,12.053,39.333,27.925,52.024,25.455,...,4.681,95.831,12.079,44.686,3.617,3.007,60.452,247.389,9.627,4.207
1,Modelo,citroen,15,102,12,99,532,220,870,404,...,22,1.934,180,944,17,17,1.716,4.992,129,36
2,AIRCROSS,citroen,0,11,1,5,60,19,89,52,...,1,277,18,130,0,1,200,589,14,1
3,AX,citroen,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,2,1,0
4,BERLINGO,citroen,0,0,0,0,4,2,5,1,...,0,6,0,3,0,0,3,20,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,CROSS TSI,wake,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
808,WAY,wake,0,0,0,0,0,1,0,0,...,0,2,1,0,0,0,0,6,0,0
809,Modelo,walk,0,1,0,0,1,0,1,0,...,0,1,0,4,0,0,2,3,0,0
810,BUGGY,walk,0,1,0,0,1,0,0,0,...,0,0,0,3,0,0,1,2,0,0
